# Phase 6 Feature Extraction — 2022 DHS Validation Tiles

**Purpose:** Extract CNN-PCA features from 2022 Sentinel-2 tiles for the 317
DHS-origin clusters (Option 3 accuracy test: 2022 imagery vs 2022 DHS labels).

**Changes from phase6_3 (2025 version):**
- `TILES_DIR` points to flat folder `Sentinel2_2022_DHS_Validation` — no ZIP files
- Tile index scans `.tif` files directly from the folder (no `zipfile` module)
- `load_tile_from_path()` replaces `load_tile_from_zip()` — opens file directly
- Filename pattern: `dhs_val_{PointID}_2022_Q{q}.tif`
- Expected: 317 PointIDs × 4 quarters = 1,268 tiles
- All preprocessing (percentile stretch, resize 224×224, VGG16 preprocess,
  PCA 4096→256) is identical to phase6_3

In [ ]:
from google.colab import drive, auth
import os
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
import pickle
from datetime import datetime

drive.mount('/content/drive')
auth.authenticate_user()

gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')
assert len(gpus) > 0, 'No GPU detected. Change runtime type to GPU.'
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# ── PATHS ───────────────────────────────────────────────────────────────────
# TILES_DIR: flat folder containing all 1,268 .tif files from phase1-1 validation export.
# No subdirectories, no ZIP files.

TILES_DIR    = '/content/drive/MyDrive/Sentinel2_2022_DHS_Validation'
MODEL_PATH   = '/content/drive/MyDrive/Models/proxy_cnn_final.keras'
PCA_PATH     = '/content/drive/MyDrive/Models/final_pca.pkl'
OUTPUT_DRIVE = '/content/drive/MyDrive/Sentinel2_2022_DHS_Validation/dynamic_features_2022_dhs_validation.csv'
CHECKPOINT   = '/content/drive/MyDrive/Sentinel2_2022_DHS_Validation/feat_extract_2022_dhs_checkpoint.csv'
GCS_OUTPUT   = 'gs://tala-sentinel2-data/features/output_2022_dhs_validation'

BATCH_SIZE   = 32

# Verify paths
for label, path in [
    ('Tiles dir', TILES_DIR),
    ('Model',     MODEL_PATH),
    ('PCA',       PCA_PATH)
]:
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {status}  {label}: {path}')

# Count TIF files — no ZIP scanning needed
tif_files = sorted([f for f in os.listdir(TILES_DIR) if f.endswith('.tif')])
print(f'\nTIF files found: {len(tif_files)}')
print(f'Expected       : {317 * 4} (317 points × 4 quarters)')
if tif_files:
    print(f'Sample names   : {tif_files[:4]}')

In [ ]:
# ── LOAD MODEL AND PCA ──────────────────────────────────────────────────────
# Identical to phase6_3 — same model, same PCA, no changes needed.

print('Loading proxy CNN feature extractor...')
full_model = load_model(MODEL_PATH)
feature_extractor = Model(
    inputs  = full_model.input,
    outputs = full_model.get_layer('feature_vector').output
)
print(f'Input  : {feature_extractor.input_shape}')
print(f'Output : {feature_extractor.output_shape}')

print('\nLoading PCA transformer...')
with open(PCA_PATH, 'rb') as f:
    pca = pickle.load(f)

N_COMPONENTS = pca.n_components_
print(f'PCA components : {N_COMPONENTS}')
print(f'PCA variance   : {pca.explained_variance_ratio_.sum():.4f}')
print('✓ Ready')

In [ ]:
# ── BUILD TILE INDEX ────────────────────────────────────────────────────────
# CHANGED from phase6_3:
#   - phase6_3 opened each ZIP and listed its internal .tif paths
#   - Here we list .tif files directly from the flat folder
#   - Filename format from phase1-1 export: dhs_val_{PointID}_2022_Q{q}.tif
#     e.g. dhs_val_1234_2022_Q1.tif
#   - Parse PointID from parts[2] and Quarter from parts[4] after split on '_'

print('Building tile index from folder...')
tile_index  = []
parse_errors = []

for fname in sorted(os.listdir(TILES_DIR)):
    if not fname.endswith('.tif'):
        continue

    # Expected: dhs_val_{PointID}_2022_Q{q}.tif
    # parts:    ['dhs', 'val', '{PointID}', '2022', 'Q{q}']
    parts = fname.replace('.tif', '').split('_')
    try:
        point_id = int(parts[2])
        quarter  = parts[4]          # Q1, Q2, Q3, Q4
        tile_index.append({
            'PointID'  : point_id,
            'Quarter'  : quarter,
            'tif_path' : os.path.join(TILES_DIR, fname)
        })
    except (IndexError, ValueError):
        parse_errors.append(fname)

if parse_errors:
    print(f'  WARNING: {len(parse_errors)} files with unexpected name format:')
    for f in parse_errors[:10]:
        print(f'    {f}')

df_index = pd.DataFrame(tile_index)
print(f'\nTotal tiles indexed   : {len(df_index)}')
print(f'Unique PointIDs       : {df_index["PointID"].nunique()}')
print(f'Quarters              : {sorted(df_index["Quarter"].unique())}')
print(f'Complete points (4/4) : '
      f'{(df_index.groupby("PointID").size() == 4).sum()}')
print(f'Incomplete points     : '
      f'{(df_index.groupby("PointID").size() < 4).sum()}')
print(f'\nSample:')
print(df_index.head(8).to_string(index=False))

In [ ]:
# ── IMAGE LOADER ────────────────────────────────────────────────────────────
# CHANGED from phase6_3:
#   - phase6_3 used rasterio zip:// URI: f'zip://{zip_path}!/{tif_internal_path}'
#   - Here we open the .tif file directly by path — no ZIP layer
#   - All preprocessing is IDENTICAL:
#     * Read bands 1,2,3 (B4=Red, B3=Green, B2=Blue)
#     * Transpose CHW → HWC
#     * NaN → 0
#     * Percentile stretch (p2–p98) → [0, 1]
#     * Resize → 224×224 bilinear
#     * VGG16 preprocess_input (scale ×255, subtract ImageNet mean)

def load_tile_from_path(tif_path, target_size=(224, 224)):
    """
    Read a GeoTIFF directly from disk, normalize, resize to 224×224,
    and apply VGG16 preprocessing.
    Returns float32 array (224, 224, 3) or None on failure.
    """
    try:
        with rasterio.open(tif_path) as src:
            # Read first 3 bands (B4=Red, B3=Green, B2=Blue)
            data = src.read([1, 2, 3]).astype(np.float32)

        # Transpose CHW → HWC
        img = np.transpose(data, (1, 2, 0))

        # Replace nodata / NaN with 0
        img = np.nan_to_num(img, nan=0.0)

        # Percentile stretch normalization (identical to phase6_3)
        p2, p98 = np.percentile(img, (2, 98))
        if p98 <= p2:
            p98 = p2 + 1e-6
        img = np.clip((img - p2) / (p98 - p2), 0.0, 1.0)

        # Resize to 224×224
        # Urban tiles (~400×400px) and rural tiles (~1000×1000px)
        # are both resized to 224×224 — same as training pipeline.
        img_tensor = tf.image.resize(
            img, target_size,
            method=tf.image.ResizeMethod.BILINEAR
        ).numpy()

        # VGG16 preprocessing (scale to [0,255], subtract ImageNet mean)
        img_vgg = img_tensor * 255.0
        img_vgg = tf.keras.applications.vgg16.preprocess_input(img_vgg)

        return img_vgg.numpy() if hasattr(img_vgg, 'numpy') else img_vgg

    except Exception as e:
        return None


# Sanity check on first tile
row0     = df_index.iloc[0]
test_img = load_tile_from_path(row0['tif_path'])
if test_img is not None:
    print(f'✓ Tile loaded successfully')
    print(f'  Shape       : {test_img.shape}')
    print(f'  Pixel range : [{test_img.min():.2f}, {test_img.max():.2f}]')
    print(f'  PointID     : {row0["PointID"]}, Quarter: {row0["Quarter"]}')
    print(f'  Path        : {row0["tif_path"]}')
else:
    print(f'✗ Tile load failed. Check path: {row0["tif_path"]}')

In [ ]:
# ── EXTRACTION LOOP ─────────────────────────────────────────────────────────
# CHANGED from phase6_3:
#   - load_tile_from_zip() → load_tile_from_path() (direct path, no zip_path arg)
#   - All other logic (batching, checkpointing, PCA transform) identical.

# Resume from checkpoint if disconnected mid-run
if os.path.exists(CHECKPOINT):
    df_done   = pd.read_csv(CHECKPOINT)
    done_keys = set(zip(
        df_done['PointID'].astype(int),
        df_done['Quarter'].astype(str)
    ))
    records   = df_done.to_dict('records')
    print(f'Resuming from checkpoint: {len(records)} tiles done')
else:
    done_keys = set()
    records   = []
    print('Starting fresh extraction')

# Filter already-done tiles
df_remaining = df_index[
    ~df_index.apply(
        lambda r: (int(r['PointID']), str(r['Quarter'])) in done_keys,
        axis=1
    )
].reset_index(drop=True)

print(f'Tiles remaining : {len(df_remaining)}')
print(f'Already done    : {len(done_keys)}')

start     = datetime.now()
batch_num = 0
n_failed  = 0

for batch_start in range(0, len(df_remaining), BATCH_SIZE):
    batch_end  = min(batch_start + BATCH_SIZE, len(df_remaining))
    batch_rows = df_remaining.iloc[batch_start:batch_end]

    imgs = []
    meta = []

    for _, row in batch_rows.iterrows():
        # CHANGED: load directly from path, no zip_path argument
        img = load_tile_from_path(row['tif_path'])
        if img is not None:
            imgs.append(img)
            meta.append((int(row['PointID']), str(row['Quarter'])))
        else:
            n_failed += 1

    if not imgs:
        batch_num += 1
        continue

    imgs_arr = np.array(imgs, dtype=np.float32)

    # CNN feature extraction → 4096-dim
    feats_4096 = feature_extractor.predict(imgs_arr, verbose=0)

    # PCA reduction → N_COMPONENTS-dim (identical to phase6_3)
    feats_pca = pca.transform(feats_4096)

    # Build records
    for j, (pid, quarter) in enumerate(meta):
        rec = {'PointID': pid, 'Quarter': quarter}
        rec.update({
            f'CNN_{k}': float(feats_pca[j][k])
            for k in range(N_COMPONENTS)
        })
        records.append(rec)
        done_keys.add((pid, quarter))

    batch_num += 1

    # Progress report every 10 batches
    if batch_num % 10 == 0:
        elapsed = (datetime.now() - start).seconds
        print(f'  [{datetime.now().strftime("%H:%M:%S")}] '
              f'Batch {batch_num} | '
              f'Done: {len(records)} | '
              f'Failed: {n_failed} | '
              f'{elapsed // 60}m {elapsed % 60}s')

    # Checkpoint to Drive every 50 batches
    if batch_num % 50 == 0:
        pd.DataFrame(records).to_csv(CHECKPOINT, index=False)
        print(f'  Checkpoint saved.')

print(f'\nExtraction complete.')
print(f'Records  : {len(records)}')
print(f'Failed   : {n_failed} tiles (load errors)')

In [ ]:
# ── SAVE AND VERIFY ─────────────────────────────────────────────────────────
# Identical structure to phase6_3 save cell.
# Output filename distinguishes this from the 2025 feature CSV.

df_final = pd.DataFrame(records).sort_values(
    ['PointID', 'Quarter']
).reset_index(drop=True)

# Save to Drive
df_final.to_csv(OUTPUT_DRIVE, index=False)
print(f'Saved to Drive: {OUTPUT_DRIVE}')

# Copy to GCS for backup
import subprocess
subprocess.run([
    'gsutil', 'cp', OUTPUT_DRIVE,
    f'{GCS_OUTPUT}/dynamic_features_2022_dhs_validation.csv'
])
print(f'Copied to GCS : {GCS_OUTPUT}')

# Verification
print(f'\n{"=" * 50}')
print('FEATURE EXTRACTION COMPLETE — 2022 DHS VALIDATION')
print(f'{"=" * 50}')
print(f'Total records    : {len(df_final)}')
print(f'Unique PointIDs  : {df_final["PointID"].nunique()}')
print(f'Quarters         : {sorted(df_final["Quarter"].unique())}')
print(f'Feature dims     : '
      f'{df_final.shape[1] - 2} (PCA-reduced, should be {N_COMPONENTS})')
print(f'Load failures    : {n_failed} tiles skipped')

# Completeness check
counts     = df_final.groupby('PointID').size()
complete   = (counts == 4).sum()
incomplete = (counts < 4).sum()
print(f'\nComplete points  (4/4 quarters) : {complete}')
print(f'Incomplete points (<4 quarters) : {incomplete}')
if incomplete > 0:
    print('Incomplete PointIDs:')
    print(counts[counts < 4].to_string())

# Spot check feature values
print(f'\nFeature value sample (first 5 dims):')
sample_cols = ['PointID', 'Quarter',
               'CNN_0', 'CNN_1', 'CNN_2', 'CNN_3', 'CNN_4']
print(df_final[sample_cols].head(4).to_string(index=False))

print(f'\nNext: run feature merge and LSTM inference on this CSV,')
print(f'then compare predictions against Actual_Wealth in')
print(f'master_cluster_summary.csv (source==DHS rows) for Option 3 accuracy.')